In [ ]:
#| default_exp components

In [ ]:
#| export

from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, stringify, VEnum, dedupe_preserve_order
from fh_matui.core import *


In [ ]:
#| export

class ContainerT(VEnum):
    """Container size options (BeerCSS).
    
    Notes:
        Most sizes alias to `responsive`. Use `expand` for full-width (`responsive max`).
    """
    xs = 'responsive'
    sm = 'responsive'
    medium = 'responsive'
    lg = 'responsive'
    xl = 'responsive'
    expand = 'responsive max'


---
description: Material Design component library for FastHTML built with BeerCSS - a drop-in replacement for MonsterUI with modern, accessible components
output-file: material-components.html
title: FastMaterial Components
---

In [ ]:
#| code-fold: true
#| eval: false

from fasthtml.jupyter import *
from IPython.display import HTML, Markdown, Image
import socket
import time
import subprocess

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=3333, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 2222
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=Theme.blue.headers(title="fastmaterial", mode="dark"))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 2222


## Buttons

In [ ]:
#| export

def NavToggleButton(target, icon='menu', **kwargs):
    """
    Create a navigation toggle button that toggles the 'max' class on the target element.
    Uses the custom toggleNav function for proper navigation rail behavior.
    
    Args:
        target (str): CSS selector for the navigation element to toggle (e.g., '#nav-id')
        icon (str): Icon name for the button (default: 'menu')
        **kwargs: Additional attributes for the Button
    
    Returns:
        Button: A button that toggles navigation using the toggleNav function
    """
    # Set default classes if not provided
    cls = kwargs.get('cls', 'circle transparent')
    
    # Create the onclick handler
    onclick = f"toggleNav('{target}'); return false;"
    
    # Update kwargs
    kwargs.update({'onclick': onclick, 'cls': cls})
    
    return Button(I(icon), **kwargs)

In [ ]:
#| export

# Button special types mapping
BUTTON_SPECIALS = {
    'primary': 'primary',
    'secondary': 'secondary', 
    'destructive': 'tertiary',
    'ghost': 'transparent border',
    'text': 'transparent',
    'link': '__link__',
    'default': 'primary'
}

def _make_button_property(tokens):
    return property(lambda self: _ButtonChain(self._tokens + tokens))

class _ButtonChain(BeerCssChain):
    pass

# Add special button properties to ButtonChain
for name, css in BUTTON_SPECIALS.items():
    tokens = css.split()
    setattr(_ButtonChain, name, _make_button_property(tokens))

# Create ButtonT instance for chaining
ButtonT = _ButtonChain()

In [ ]:
#| code-fold: true
#| eval: false

def ex_buttons(): 

    return  Grid(
        Button("Default", cls=ButtonT.default),
        Button("Primary", cls=ButtonT.primary),
        Button("Secondary", cls=ButtonT.secondary),
        Button("Danger", cls=ButtonT.destructive),
        Button("Text", cls=ButtonT.text),
        Button("Ghost", cls=ButtonT.ghost),
        Button("Small Primary", cls=ButtonT.primary.small),
        Button("Large Secondary", cls=ButtonT.secondary.large.elevate)
        
    )

preview(ex_buttons())

## Anchor

In [ ]:
#| export

class _AnchorChain(BeerCssChain):
    pass

ANCHOR_SPECIALS = {
    'muted': 'grey-text',
    'text': '',
    'reset': 'no-underline',
    'primary': 'primary-text link',
    'classic': 'link',
    'inverse': 'inverse-link'
}

def _make_anchor_property(tokens):
    return property(lambda self: _AnchorChain(self._tokens + tokens))

for name, css in ANCHOR_SPECIALS.items():
    tokens = css.split() if css else []
    setattr(_AnchorChain, name, _make_anchor_property(tokens))

AT = _AnchorChain()

In [ ]:
#| code-fold: true
#| eval: false

def ex_links(): 
    return Div(
        A('Default Link'),
        A('Muted Link', cls=AT.muted),
        A('Text Link',  cls=AT.text),
        A('Reset Link', cls=AT.reset),
        A('Primary Link', cls=AT.primary),
        A('Classic Link', cls=AT.classic),)

preview(ex_links())

## Grid

In [ ]:
#| export

# Spacing enums for autocomplete while keeping raw BeerCSS tokens
class SpaceT(VEnum):
    """BeerCSS spacing helper tokens."""
    no_space = 'no-space'
    small_space = 'small-space'
    medium_space = 'medium-space'
    large_space = 'large-space'
    space = 'space'


class GridSpanT(VEnum):
    """BeerCSS 12-col grid span tokens (s/m/l breakpoints)."""
    # Small breakpoint
    s1 = 's1'
    s2 = 's2'
    s3 = 's3'
    s4 = 's4'
    s5 = 's5'
    s6 = 's6'
    s7 = 's7'
    s8 = 's8'
    s9 = 's9'
    s10 = 's10'
    s11 = 's11'
    s12 = 's12'

    # Medium breakpoint
    m1 = 'm1'
    m2 = 'm2'
    m3 = 'm3'
    m4 = 'm4'
    m5 = 'm5'
    m6 = 'm6'
    m7 = 'm7'
    m8 = 'm8'
    m9 = 'm9'
    m10 = 'm10'
    m11 = 'm11'
    m12 = 'm12'

    # Large breakpoint
    l1 = 'l1'
    l2 = 'l2'
    l3 = 'l3'
    l4 = 'l4'
    l5 = 'l5'
    l6 = 'l6'
    l7 = 'l7'
    l8 = 'l8'
    l9 = 'l9'
    l10 = 'l10'
    l11 = 'l11'
    l12 = 'l12'


def _has_space_token(tokens):
    space_tokens = {'space', 'no-space', 'small-space', 'medium-space', 'large-space'}
    return any(t in space_tokens for t in tokens)


def GridCell(*c, span=(), cls='', **kwargs):
    """Wrap content as a BeerCSS grid *cell* (direct child of `.grid`).
    
    `span` can be a string (e.g. 's12 m6 l4') or a tuple of strings/enums,
    e.g. `(GridSpanT.s12, GridSpanT.m6, GridSpanT.l4)`.
    """
    cell_cls = []
    cell_cls.extend(normalize_tokens(span))
    cell_cls.extend(normalize_tokens(cls))
    cell_cls = [t for t in cell_cls if t]
    return Div(*c, cls=stringify(dedupe_preserve_order(cell_cls)), **kwargs)


def ResponsiveGrid(*cells, space=SpaceT.medium_space, cls: str = '', **kwargs):
    """BeerCSS grid wrapper that preserves the 'direct children are cells' rule.
    
    Pass `GridCell(...)` children (or any already-wrapped `.grid` cells).
    """
    cls_tokens = normalize_tokens(cls)
    grid_cls = ['grid']
    if space and not _has_space_token(cls_tokens):
        grid_cls.extend(normalize_tokens(space))
    grid_cls.extend(cls_tokens)
    grid_cls = [t for t in grid_cls if t]
    return Div(*cells, cls=stringify(dedupe_preserve_order(grid_cls)), **kwargs)


def DivHStacked(*c, cls='', **kwargs):
    """MonsterUI-compatible horizontal stack using BeerCSS tokens."""
    cls_tokens = normalize_tokens(cls)
    tokens = []
    if 'grid' not in cls_tokens:
        tokens.extend(['row', 'middle-align'])
    if not _has_space_token(cls_tokens):
        tokens.append(SpaceT.medium_space)
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    return Div(*c, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)


def DivVStacked(*c, cls='', **kwargs):
    """MonsterUI-compatible vertical stack using BeerCSS tokens.
    
    If you pass `cls='grid ...'`, this acts as a pure BeerCSS grid wrapper (no 'column' token),
    so direct-child grid cells (`s12 m6 l3`, etc.) behave correctly.
    """
    cls_tokens = normalize_tokens(cls)
    tokens = []
    if 'grid' not in cls_tokens:
        tokens.append('column')
    if not _has_space_token(cls_tokens):
        tokens.append(SpaceT.medium_space)
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    return Div(*c, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)


def DivFullySpaced(*c, cls='', **kwargs):
    """MonsterUI-compatible fully-spaced row using BeerCSS tokens.
    
    Uses BeerCSS `max` as flexible spacers so children stretch to the far ends.
    """
    cls_tokens = normalize_tokens(cls)
    tokens = []
    if 'grid' not in cls_tokens:
        tokens.extend(['row', 'middle-align'])
    if not _has_space_token(cls_tokens):
        tokens.append(SpaceT.no_space)
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    base = list(c)
    if 'grid' not in cls_tokens and len(base) > 1:
        spaced_children = []
        for i, child in enumerate(base):
            spaced_children.append(child)
            if i != len(base) - 1:
                spaced_children.append(Div(cls='max'))
        base = spaced_children
    return Div(*base, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)


def DivCentered(*c, cls='', **kwargs):
    """MonsterUI-compatible centered stack using BeerCSS tokens."""
    cls_tokens = normalize_tokens(cls)
    tokens = ['center-align']
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    return DivVStacked(*c, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)


def DivLAligned(*c, cls='', **kwargs):
    """MonsterUI-compatible left-aligned row using BeerCSS tokens."""
    cls_tokens = normalize_tokens(cls)
    tokens = ['left-align']
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    return DivHStacked(*c, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)


def DivRAligned(*c, cls='', **kwargs):
    """MonsterUI-compatible right-aligned row using BeerCSS tokens."""
    cls_tokens = normalize_tokens(cls)
    tokens = ['right-align']
    tokens.extend(cls_tokens)
    tokens = [t for t in tokens if t]
    return DivHStacked(*c, cls=stringify(dedupe_preserve_order(tokens)), **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_grid():
    return Grid(
        Div(
            P("Column 1 Item 1"), 
            P("Column 1 Item 2"), 
            P("Column 1 Item 3")),
        Div(
            P("Column 2 Item 1"), 
            P("Column 2 Item 2"), 
            P("Column 2 Item 2")),
        Div(
            P("Column 3 Item 1"), 
            P("Column 3 Item 2"), 
            P("Column 3 Item 3")))


preview(ex_grid())

In [ ]:
#| code-fold: true
#| eval: false

def ex_product_grid():
    products = [
        {"name": "Laptop", "price": "$999", "img": "https://picsum.photos/200/100?random=1"},
        {"name": "Smartphone", "price": "$599", "img": "https://picsum.photos/200/100?random=2"},
        {"name": "Headphones", "price": "$199", "img": "https://picsum.photos/200/100?random=3"},
        {"name": "Smartwatch", "price": "$299", "img": "https://picsum.photos/200/100?random=4"},
        {"name": "Tablet", "price": "$449", "img": "https://picsum.photos/200/100?random=5"},
        {"name": "Camera", "price": "$799", "img": "https://picsum.photos/200/100?random=6"},
    ]
    
    product_cards = [
        Card(
            Img(src=p["img"], alt=p["name"], style="width:100%; height:100px; object-fit:cover;"),
            H4(p["name"], cls="mt-2"),
            P(p["price"]),
            Button("Add to Cart", cls=(ButtonT.primary, "mt-2"))
        ) for p in products
    ]
    
    return Grid(*product_cards, cols_lg=3)

preview(ex_product_grid())

In [ ]:
#| code-fold: true
#| eval: false

def ex_l_aligned_div():
    return DivLAligned(
        Img(src="https://picsum.photos/100/100?random=1", style="max-width: 100px;"),
        H4("Left Aligned Title"),
        P("Some text that's left-aligned with the title and image.")
    )

preview(ex_l_aligned_div())

In [ ]:
#| code-fold: true
#| eval: false

def ex_v_stacked_div():
    return DivVStacked(
        H2("Vertical Stack"),
        P("First paragraph in the stack"),
        P("Second paragraph in the stack"),
        Button("Action Button", cls=ButtonT.secondary)
    )

preview(ex_v_stacked_div())

In [ ]:
#| code-fold: true
#| eval: false

def ex_h_stacked_div():
    return DivHStacked(
        Div(H4("Column 1"), P("Content for column 1")),
        Div(H4("Column 2"), P("Content for column 2")),
        Div(H4("Column 3"), P("Content for column 3"))
    )

preview(ex_h_stacked_div())

In [ ]:
#| code-fold: true
#| eval: false

def ex_r_aligned_div():
    return DivRAligned(
        Button("Action", cls=ButtonT.primary),
        P("Right-aligned text"),
        Img(src="https://picsum.photos/100/100?random=3", style="max-width: 100px;")
    )

preview(ex_r_aligned_div())

In [ ]:
#| code-fold: true
#| eval: false

def ex_centered_div():
    return DivCentered(
        H3("Centered Title"),
        P("This content is centered both horizontally and vertically.")
    )

preview(ex_centered_div())

In [ ]:
#| code-fold: true
#| eval: false

def ex_fully_spaced_div():
    return DivFullySpaced(
        Button("Left", cls=ButtonT.primary),
        Button("Center", cls=ButtonT.secondary),
        Button("Right", cls=ButtonT.destructive)
    )

preview(ex_fully_spaced_div())

## Icon

In [ ]:
#| export

def Icon(icon: str,
         size: str = None,
         fill: bool = False,
         cls = (),
         **kwargs):
    """
    BeerCSS Material Icon helper with MonsterUI-compatible signature.
    
    Args:
        icon: Material icon name (e.g., 'home', 'menu', 'settings')
        size: Optional size ('tiny', 'small', 'medium', 'large', 'extra')
        fill: Whether icon should be filled (default: False)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <i> element with Material icon
    
    Examples:
        Icon('home')
        Icon('menu', size='large')
        Icon('favorite', fill=True)
        Icon('settings', cls='primary-text')
    """
    # Build class list
    icon_cls = []
    if size:
        icon_cls.append(size)
    if fill:
        icon_cls.append('fill')
    if cls:
        icon_cls.extend(normalize_tokens(cls))
    
    cls_str = ' '.join(icon_cls) if icon_cls else None
    
    if cls_str:
        return I(icon, cls=cls_str, **kwargs)
    return I(icon, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_icon():
    return Grid(
           Icon('menu', size='large'),     
           Icon('settings', cls='primary-text'),
               Icon('favorite', fill=True)
    )
  
preview(ex_icon())

## NavBar

In [ ]:
#| export

def NavBar(*children, 
           brand=None,
           sticky=False,
           cls='',
           **kwargs):
    """
    BeerCSS horizontal navigation bar with full spacing.
    
    Args:
        *children: Navigation items (A() links, buttons, etc.)
        brand: Brand/logo component (H3, Img, Div, etc.) positioned at start
        sticky: Whether navbar sticks to top while scrolling (default: False)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <nav> element with horizontal layout and surface-container background
    
    Examples:
        # Simple navbar with links
        NavBar(
            A("Page1", href='/rt1'),
            A("Page2", href='/rt2'),
            A("Page3", href='/rt3')
        )
        
        # Navbar with brand and sticky (brand on left, links on right)
        NavBar(
            A("Home", href='/'),
            A("About", href='/about'),
            A("Contact", href='/contact'),
            brand=H3('My Blog'),
            sticky=True
        )
    """
    # If brand is provided, use DivFullySpaced for layout
    if brand:
        # Brand on far left, links grouped on far right
        content = DivFullySpaced(
            brand,
            DivHStacked(*children),  # Links grouped horizontally on right
            cls='padding'
        )
    else:
        # No brand, just horizontal stack of links
        content = DivHStacked(*children, cls='padding')
    
    # Apply sticky class and surface-container background (no 'max' - that's for nav rails)
    nav_cls = f"{'sticky top' if sticky else ''} surface-container {cls}".strip()
    
    return Nav(content, cls=nav_cls, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_navbar1():
    return NavBar(A("Page1",href='/rt1'),
                  A("Page2",href='/rt2'),
                  A("Page3",href='/rt3'),
                  brand=H3('My Blog'))

preview(ex_navbar1())

## Modal Dialog

In [ ]:
#| export

def Modal(*c, id=None, footer=None, active=False, overlay=True, cls=(), **kwargs):
    """
    BeerCSS Modal Dialog component.
    
    Args:
        *c: Modal content (title, body, etc.)
        id: Modal ID for data-ui targeting
        footer: Footer content (auto-wrapped in Nav if not already)
        active: Whether modal starts active
        overlay: Whether to include overlay
        cls: Additional CSS classes for dialog
        **kwargs: Additional HTML attributes
    
    Returns:
        Dialog element (or list with overlay if overlay=True)
    
    Example:
        Modal(
            ModalTitle("Custom overlay"),
            ModalBody("Some text here"),
            footer=ModalFooter(
                ModalCancel(modal_id="my-modal"),
                ModalConfirm(modal_id="my-modal")
            ),
            id="my-modal"
        )
    """
    modal_cls = normalize_tokens(cls)
    if active: 
        modal_cls.append('active')
    
    children = list(c)
    if footer:
        if hasattr(footer, 'tag') and footer.tag == 'nav': 
            children.append(footer)
        else: 
            children.append(Nav(*(footer if is_listy(footer) else [footer])))
    
    cls_str = ' '.join(modal_cls) if modal_cls else None
    
    # Create the dialog
    dialog = Dialog(*children, id=id, cls=cls_str, **kwargs)
    
    if overlay:
        # Return overlay + dialog as separate elements that BeerCSS can manage
        overlay_cls = "overlay blur"
        if active:
            overlay_cls += " active"
            
        # Return as a list - both elements need to be at same DOM level
        return [
            Div(cls=overlay_cls),
            dialog
        ]
    
    return dialog

def ModalButton(text: str, id: str, icon: str = None, cls=(), **kwargs):
    """
    Button that opens a modal using BeerCSS data-ui attribute.
    
    Args:
        text: Button text
        id: ID of modal to open (without #)
        icon: Optional icon name
        cls: Button styles
        **kwargs: Additional HTML attributes
    
    Example:
        ModalButton("Open Modal", "my-modal", cls=ButtonT.primary)
    """
    kwargs["data_ui"] = f"#{id}"
    return Button(text, icon=icon, cls=cls, **kwargs)

def ModalCancel(text="Cancel", modal_id=None, cls=(), **kwargs):
    """
    Cancel button for modals - automatically includes data-ui to close modal.
    
    Args:
        text: Button text (default "Cancel")
        modal_id: ID of modal to close (without #)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    """
    cancel_cls = normalize_tokens(cls)
    cancel_cls.extend(['transparent', 'link'])
    if modal_id:
        kwargs["data_ui"] = f"#{modal_id}"
    return Button(text, cls=' '.join(cancel_cls), **kwargs)

def ModalConfirm(text="Confirm", modal_id=None, cls=(), **kwargs):
    """
    Confirm button for modals - automatically includes data-ui to close modal.
    
    Args:
        text: Button text (default "Confirm") 
        modal_id: ID of modal to close (without #)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    """
    confirm_cls = normalize_tokens(cls)
    confirm_cls.extend(['transparent', 'link'])
    if modal_id:
        kwargs["data_ui"] = f"#{modal_id}"
    return Button(text, cls=' '.join(confirm_cls), **kwargs)

def ModalTitle(*c, cls=(), **kwargs):
    """Modal title component using H5 (BeerCSS default)."""
    return H5(*c, cls=stringify(cls), **kwargs)

def ModalBody(*c, cls=(), **kwargs):
    """Optional modal body wrapper for complex content layout."""
    return Div(*c, cls=stringify(cls), **kwargs)

def ModalFooter(*c, cls=(), **kwargs):
    """
    Modal footer component using Nav element for action buttons.
    Automatically includes right-align no-space classes for proper button layout.
    """
    footer_cls = normalize_tokens(cls)
    footer_cls.extend(['right-align', 'no-space'])
    return Nav(*c, cls=' '.join(footer_cls), **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def test_modal():
    """Test modal using convenience methods - no data-ui needed!"""
    return Div(
        # Trigger button
        Button("Open Modal", data_ui="#fix-modal", cls="primary"),
        
        # Modal with simplified helper functions
        *Modal(
            ModalTitle("Custom overlay"),
            ModalBody("Some text here"),
            footer=ModalFooter(
                ModalCancel(modal_id="fix-modal"),
                ModalConfirm(modal_id="fix-modal")
            ),
            id="fix-modal"
        )
    )

preview(test_modal())

## Label Input

In [ ]:
#| export

def Field(*c, 
          label: bool = False,
          prefix: bool = False,
          suffix: bool = False,
          cls = '',
          **kwargs):
    """
    BeerCSS field wrapper for inputs with default styling.
    
    Args:
        *c: Input, label, and icon elements
        label: Add 'label' class for floating label behavior
        prefix: Add 'prefix' class for prefix icons
        suffix: Add 'suffix' class for suffix icons
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <div class="field small round border"> wrapper with appropriate classes
    
    Examples:
        # Simple field
        Field(Input(type="text"))
        
        # Field with floating label
        Field(
            Input(type="text", placeholder=" "),
            Label("Username"),
            label=True
        )
        
        # Field with prefix icon
        Field(
            I('search'),
            Input(type="text", placeholder=" "),
            Label("Search"),
            label=True,
            prefix=True
        )
    """
    # Default classes for good styling
    field_cls = ['field', 'small', 'round', 'border']
    if label:
        field_cls.append('label')
    if prefix:
        field_cls.append('prefix')
    if suffix:
        field_cls.append('suffix')
    
    cls_str = f"{' '.join(field_cls)} {cls}".strip()
    return Div(*c, cls=cls_str, **kwargs)

def LabelInput(label: str,
               id: str = None,
               placeholder: str = None,
               input_type: str = 'text',
               prefix_icon: str = None,
               suffix_icon: str = None,
               value: str = None,
               cls = '',
               **kwargs):
    """
    Labeled input field with MonsterUI-compatible signature.
    Creates a BeerCSS field with floating label and optional icons.
    
    Args:
        label: Label text
        id: Input ID (auto-generated from label if not provided)
        placeholder: Placeholder text (defaults to " " for label animation)
        input_type: Input type (text, email, password, etc.)
        prefix_icon: Material icon name for prefix
        suffix_icon: Material icon name for suffix
        value: Initial input value
        cls: Additional CSS classes for field wrapper
        **kwargs: Additional HTML attributes for input
    
    Returns:
        Field with input, label, and optional icons
    
    Examples:
        # Simple labeled input
        LabelInput("Email", input_type="email")
        
        # With prefix icon
        LabelInput("Search", prefix_icon="search")
        
        # With custom ID and value
        LabelInput("Username", id="user", value="john_doe")
    """
    # Auto-generate ID from label if not provided
    if not id:
        id = label.lower().replace(' ', '-')
    
    children = []
    
    # Add prefix icon if provided
    if prefix_icon:
        children.append(I(prefix_icon))
    
    # Build input attributes
    input_attrs = {
        'type': input_type,
        'id': id,
        'name': id,
        'placeholder': placeholder if placeholder is not None else " "
    }
    if value is not None:
        input_attrs['value'] = value
    input_attrs.update(kwargs)
    
    # Add input
    children.append(Input(**input_attrs))
    
    # Add label (for attribute links to input id)
    children.append(Label(label, fr=id))
    
    # Add suffix icon if provided
    if suffix_icon:
        children.append(I(suffix_icon))
    
    # Return wrapped field with appropriate classes
    return Field(
        *children,
        label=True,
        prefix=bool(prefix_icon),
        suffix=bool(suffix_icon),
        cls=cls
    )


In [ ]:
#| code-fold: true
#| eval: false

def ex_input(): 
    return Div(
        Br(),
        #Input is not implemented
        LabelInput(label="Input", id='myid'))


preview(ex_input())

## Form Label

In [ ]:
#| export

def LabelInput(label: str,
               id: str = None,
               placeholder: str = None,
               input_type: str = 'text',
               prefix_icon: str = None,
               suffix_icon: str = None,
               value: str = None,
               cls = '',
               **kwargs):
    """
    Labeled input field with MonsterUI-compatible signature.
    Creates a BeerCSS field with floating label and optional icons.
    
    Args:
        label: Label text
        id: Input ID (auto-generated from label if not provided)
        placeholder: Placeholder text (defaults to " " for label animation)
        input_type: Input type (text, email, password, etc.)
        prefix_icon: Material icon name for prefix
        suffix_icon: Material icon name for suffix
        value: Initial input value
        cls: Additional CSS classes for field wrapper
        **kwargs: Additional HTML attributes for input
    
    Returns:
        Field with input, label, and optional icons
    
    Examples:
        # Simple labeled input
        LabelInput("Email", input_type="email")
        
        # With prefix icon
        LabelInput("Search", prefix_icon="search")
        
        # With custom ID and value
        LabelInput("Username", id="user", value="john_doe")
    """
    # Auto-generate ID from label if not provided
    if not id:
        id = label.lower().replace(' ', '-')
    
    children = []
    
    # Add prefix icon if provided
    if prefix_icon:
        children.append(I(prefix_icon))
    
    # Build input attributes
    input_attrs = {
        'type': input_type,
        'id': id,
        'name': id,
        'placeholder': placeholder if placeholder is not None else " "
    }
    if value is not None:
        input_attrs['value'] = value
    input_attrs.update(kwargs)
    
    # Add input
    children.append(Input(**input_attrs))
    
    # Add label (for attribute links to input id)
    children.append(Label(label, fr=id))
    
    # Add suffix icon if provided
    if suffix_icon:
        children.append(I(suffix_icon))
    
    # Return wrapped field with appropriate classes
    return Field(
        *children,
        label=True,
        prefix=bool(prefix_icon),
        suffix=bool(suffix_icon),
        cls=cls
    )

def FormLabel(*c, cls=(), **kwargs):
    """
    Standalone form label with MonsterUI-compatible signature.
    
    Args:
        *c: Label text or child elements
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes (include 'fr' for for-attribute)
    
    Returns:
        <label> element
    
    Example:
        FormLabel("Username", fr="username-input")
    """
    cls_str = stringify(cls) if cls else None
    if cls_str:
        return Label(*c, cls=cls_str, **kwargs)
    return Label(*c, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_form_label(): 
    return Div(
        Br(),
        FormLabel("Username", fr="username-input"))


preview(ex_form_label())

## Checkbox

In [ ]:
#| export

def CheckboxX(*c, cls=(), **kwargs):
    """
    BeerCSS checkbox with MonsterUI-compatible signature.
    
    Args:
        *c: Label text or child elements to display next to checkbox
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes for input (checked, disabled, name, value, etc.)
    
    Returns:
        <label class="checkbox"> wrapping input and span
    
    Examples:
        CheckboxX("Accept terms")
        CheckboxX("Enabled", checked=True)
        CheckboxX("Option 1", name="options", value="opt1")
    """
    # Extract text content
    label_text = stringify(c) if c else ""
    
    # Build checkbox structure: <label class="checkbox"><input><span>text</span></label>
    checkbox_cls = ['checkbox']
    if cls:
        checkbox_cls.extend(normalize_tokens(cls))
    
    cls_str = stringify(checkbox_cls)
    
    return Label(
        Input(type='checkbox', **kwargs),
        Span(label_text) if label_text else Span(),
        cls=cls_str
    )



In [ ]:
#| code-fold: true
#| eval: false

def ex_checkbox(): 
    return Div(
        CheckboxX(),
        CheckboxX("Accept terms"),
        CheckboxX("Enabled", checked=True),
        CheckboxX("Option 1", name="options", value="opt1")
    )
        
preview(ex_checkbox())

##  Radio

In [ ]:
#| export

def Radio(*c, cls=(), **kwargs):
    """
    BeerCSS radio button with MonsterUI-compatible signature.
    
    Args:
        *c: Label text or child elements to display next to radio
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes for input (checked, disabled, name, value, etc.)
    
    Returns:
        <label class="radio"> wrapping input and span
    
    Examples:
        Radio("Option 1", name="group1", value="opt1")
        Radio("Option 2", name="group1", value="opt2", checked=True)
    """
    # Extract text content
    label_text = stringify(c) if c else ""
    
    # Build radio structure: <label class="radio"><input><span>text</span></label>
    radio_cls = ['radio']
    if cls:
        radio_cls.extend(normalize_tokens(cls))
    
    cls_str = stringify(radio_cls)
    
    return Label(
        Input(type='radio', **kwargs),
        Span(label_text) if label_text else Span(),
        cls=cls_str
    )



In [ ]:
#| code-fold: true
#| eval: false

def ex_Radio(): 
    return Div(
        Br(),
         Radio("Option 1", name="group1", value="opt1"),
        Radio("Option 2", name="group1", value="opt2", checked=True)
    )

preview(ex_Radio())

## Switch

In [ ]:
#| export

def Switch(*c, cls=(), **kwargs):
    """
    BeerCSS toggle switch with MonsterUI-compatible signature.
    
    Args:
        *c: Label text or child elements to display next to switch
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes for checkbox input (checked, disabled, name, etc.)
    
    Returns:
        <label class="switch"> wrapping label span, checkbox input, and toggle span
    
    Examples:
        Switch("Dark Mode", checked=True)
        Switch("Enable notifications", name="dark_mode", checked=False)
    """
    # Extract text content
    label_text = stringify(c) if c else ""
    
    # Build switch structure: <label class="switch"><span style="margin-right">Label</span><input><span></span></label>
    switch_cls = ['switch']
    if cls:
        switch_cls.extend(normalize_tokens(cls))
    
    cls_str = stringify(switch_cls)
    
    # Label span comes first (before the toggle), with right margin for spacing
    children = []
    if label_text:
        children.append(Span(label_text, style="margin-right: 0.5rem"))
    
    children.extend([
        Input(type='checkbox', **kwargs),
        Span()  # Visual toggle element
    ])
    
    return Label(*children, cls=cls_str)


In [ ]:
#| code-fold: true
#| eval: false

def ex_switch(): 
    return Div(
             Switch("Enable notifications", name="dark_mode", checked=False)
    )
preview(ex_switch())

# The more commonn would be space out switch and text.  TODO

## Text Area

In [ ]:
#| export

def TextArea(*c, cls=(), **kwargs):
    """
    BeerCSS textarea with MonsterUI-compatible signature.
    Wrapped in field for consistent styling.
    
    Args:
        *c: Initial text content
        cls: Additional CSS classes for field wrapper
        **kwargs: Additional HTML attributes for textarea (rows, placeholder, name, etc.)
    
    Returns:
        Field-wrapped <textarea>
    
    Examples:
        TextArea(placeholder="Enter description...")
        TextArea("Initial text", rows=5)
        TextArea(placeholder=" ")  # For floating label
    """
    # Extract content
    content = stringify(c) if c else ""
    
    # Create textarea
    textarea = Textarea(content, **kwargs) if content else Textarea(**kwargs)
    
    # Wrap in field with default styling
    return Field(textarea, cls=cls)


In [ ]:
#| code-fold: true
#| eval: false

def ex_TextArea(): 
    return Div(
        Br(),
        TextArea(placeholder="Enter description..."),
        TextArea("Initial text", rows=5),  # this does not expand as it should. 
        TextArea(placeholder=" ") 
    )

preview(ex_TextArea())

## Range

In [ ]:
#| export

def Range(*c, min=None, max=None, step=None, cls=(), **kwargs):
    """
    BeerCSS range slider with MonsterUI-compatible signature.
    
    Args:
        *c: Not used (for signature compatibility)
        min: Minimum value
        max: Maximum value
        step: Step increment
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes (value, name, etc.)
    
    Returns:
        <label class="slider"> wrapping input[type="range"] and span with CSS variables
    
    Examples:
        Range(min=0, max=100, value=50)
        Range(min=1, max=10, step=1)
    """
    # Default values
    min_val = min if min is not None else 0
    max_val = max if max is not None else 100
    value = kwargs.get('value', min_val)
    
    # Calculate percentage for two-tone effect
    if max_val != min_val:
        percentage = ((float(value) - float(min_val)) / (float(max_val) - float(min_val))) * 100
    else:
        percentage = 0
    
    # Build input attributes
    input_attrs = {'type': 'range'}
    if min is not None:
        input_attrs['min'] = min
    if max is not None:
        input_attrs['max'] = max
    if step is not None:
        input_attrs['step'] = step
    
    # Add oninput handler to dynamically update CSS variables
    # --_start to --_end defines the FILLED (darker) portion
    # So filled portion is 0% to percentage, unfilled is percentage to 100%
    input_attrs['oninput'] = """
        const val = this.value;
        const min = this.min || 0;
        const max = this.max || 100;
        const percentage = ((val - min) / (max - min)) * 100;
        this.parentElement.style.setProperty('--_start', '0%');
        this.parentElement.style.setProperty('--_end', percentage + '%');
    """.strip()
    
    input_attrs.update(kwargs)
    
    # Build slider structure with CSS variables for two-tone effect
    slider_cls = ['slider']
    if cls:
        slider_cls.extend(normalize_tokens(cls))
    
    cls_str = stringify(slider_cls)
    
    # Apply initial CSS custom properties for BeerCSS two-tone styling
    # --_start to --_end is the filled (darker) range
    style = f"--_start: 0%; --_end: {percentage:.1f}%;"
    
    return Label(
        Input(**input_attrs),
        Span(),
        cls=cls_str,
        style=style
    )


In [ ]:
#| code-fold: true
#| eval: false

def ex_range(): 
    return Div(
               Range(value=25),   Range(min=0, max=100, value=50)
    )
preview(ex_range())

## Select

In [ ]:
#| export

def Select(*items, 
           value='', 
           placeholder='Select...', 
           prefix_icon=None,
           name='',
           cls=(),
           **kwargs):
    """
    BeerCSS menu-based select with beautiful styling (default).
    Uses field + readonly input + menu pattern for rich dropdown experience.
    
    Args:
        *items: Menu items - can be strings, Li elements, or mixed content
        value: Currently selected value (displayed in input)
        placeholder: Placeholder text when no value selected
        prefix_icon: Material icon name for prefix (optional)
        name: Input name for form submission
        cls: Additional CSS classes for field wrapper
        **kwargs: Additional HTML attributes
    
    Returns:
        Field with readonly input and menu dropdown
    
    Examples:
        Select("Home", "About", "Contact", value="Home")
        Select("Option 1", "Option 2", "Option 3", prefix_icon="list")
        
        # With sections and icons
        Select(
            Li(Label("Section"), cls="transparent"),
            Li("Item 1"),
            Li("Item 2"),
            Hr(),
            Li(I("star", cls="tiny"), Div("Special", cls="max")),
            value="Item 1",
            prefix_icon="menu"
        )
    """
    # Build menu items
    menu_items = []
    for item in items:
        if isinstance(item, str):
            menu_items.append(Li(item))
        else:
            menu_items.append(item)
    
    # Build field children
    children = []
    
    # Add prefix icon if provided
    if prefix_icon:
        children.append(I(prefix_icon))
    
    # Add readonly input showing current value
    input_attrs = {
        'value': value,
        'readonly': True,
        'placeholder': placeholder if placeholder else ' '
    }
    if name:
        input_attrs['name'] = name
    input_attrs.update(kwargs)
    
    children.append(Input(**input_attrs))
    
    # Add dropdown arrow icon
    children.append(I('arrow_drop_down'))
    
    # Add menu with items
    children.append(Menu(*menu_items))
    
    # Build field classes
    field_cls = ['field', 'fill', 'round']
    if prefix_icon:
        field_cls.append('prefix')
    field_cls.append('suffix')  # For arrow icon
    
    if cls:
        field_cls.extend(normalize_tokens(cls))
    
    cls_str = stringify(field_cls)
    
    return Div(*children, cls=cls_str)


In [ ]:
#| code-fold: true
#| eval: false

def ex_select(): 
    return Div(
             Select("Home", "About", "Contact", value="Home"),
        Select("Option 1", "Option 2", "Option 3", prefix_icon="list"),
        
        # With sections and icons
        Select(
            Li(Label("Section"), cls="transparent"),
            Li("Item 1"),
            Li("Item 2"),
            Hr(),
            Li(I("star", cls="tiny"), Div("Special", cls="max")),
            value="Item 1",
            prefix_icon="menu"
        )
    )
preview(ex_select())

### Fieldset

In [ ]:
def Fieldset(*c, cls=(), **kwargs):
    """
    Form fieldset with MonsterUI-compatible signature.
    
    Args:
        *c: Child elements (form controls, legend, etc.)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <fieldset> element
    
    Example:
        Fieldset(
            Legend("Personal Info"),
            LabelInput("Name"),
            LabelInput("Email")
        )
    """
    cls_str = stringify(cls) if cls else None
    if cls_str:
        return fc.Fieldset(*c, cls=cls_str, **kwargs)
    return fc.Fieldset(*c, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_fs(): 
    return Div(
             Fieldset(
            Legend("Personal Info"),
            LabelInput("Name"),
            LabelInput("Email")
        )
    )
preview(ex_fs())

### Legend

In [ ]:

def Legend(*c, cls=(), **kwargs):
    """
    Fieldset legend/caption with MonsterUI-compatible signature.
    
    Args:
        *c: Legend text or child elements
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <legend> element
   
    Example:
        Legend("Account Settings")
    """
    cls_str = stringify(cls) if cls else None
    if cls_str:
        return fc.Legend(*c, cls=cls_str, **kwargs)
    return fc.Legend(*c, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_l():
    return Div(
             Legend("Account Settings")
    )
preview(ex_l())

## Progress

In [ ]:
#| export

def Progress(*c, value='', max='100', cls=(), **kwargs):
    """
    Progress bar with MonsterUI-compatible signature.
    
    Args:
        *c: Not used (for signature compatibility)
        value: Current progress value
        max: Maximum value (default: '100')
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <progress> element
    
    Examples:
        Progress(value=50, max=100)
        Progress(value=75)
    """
    progress_attrs = {}
    if value:
        progress_attrs['value'] = value
    if max:
        progress_attrs['max'] = max
    progress_attrs.update(kwargs)
    
    cls_str = stringify(cls) if cls else None
    if cls_str:
        return fc.Progress(*c, cls=cls_str, **progress_attrs)
    return fc.Progress(*c, **progress_attrs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_progress():
    return Div(
                Progress(value=50, max=100),    Progress(value=75)
    )
preview(ex_progress())

## Loading Indicator

In [ ]:
#| export

def LoadingIndicator(size='medium', cls='', **kwargs):
    """
    BeerCSS circular loading indicator.
    
    Args:
        size: Size variation ('small', 'medium', 'large')
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes (id for hx_indicator)
    
    Returns:
        <progress class="circle"> element
    
    Examples:
        LoadingIndicator()
        LoadingIndicator(size='small', id='loading')
        # With HTMX: hx_indicator='#loading'
    """
    size_cls = size if size else 'medium'
    progress_cls = f"circle {size_cls} {cls}".strip()
    return fc.Progress(cls=progress_cls, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_progress():
    return Div(
               LoadingIndicator(),
        LoadingIndicator(size='small', id='loading')
    )
preview(ex_progress())

## Tables

In [ ]:
#| export

def Table(*c,
          cls = 'border',
          **kwargs):
    """
    BeerCSS table component with MonsterUI-compatible signature.
    
    Args:
        *c: Table children (Thead, Tbody, Tfoot)
        cls: CSS classes (default 'border', can add 'stripes')
        **kwargs: Additional HTML attributes
        
    Returns:
        Table element with BeerCSS styling
        
    Example:
        Table(
            Thead(Tr(Th("Name"), Th("Email"))),
            Tbody(Tr(Td("John"), Td("john@example.com"))),
            cls="border stripes"
        )
    """
    cls_str = stringify(cls) if cls else 'border'
    return fc.Table(*c, cls=cls_str, **kwargs)


def Td(*c,
       shrink = False,
       expand = False,
       cls = (),
       **kwargs):
    """
    Table cell with MonsterUI-compatible signature.
    
    Args:
        *c: Cell content
        shrink: Minimize cell width
        expand: Maximize cell width
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
        
    Returns:
        Td element
    """
    cls_str = stringify(cls) if cls else ''
    if shrink:
        cls_str += ' no-wrap'
        existing_style = kwargs.get('style', '').strip()
        if existing_style and not existing_style.endswith(';'):
            existing_style += ';'
        kwargs['style'] = f"{existing_style} width: 1%;".strip()
    if expand:
        cls_str += ' no-wrap'
        existing_style = kwargs.get('style', '').strip()
        if existing_style and not existing_style.endswith(';'):
            existing_style += ';'
        kwargs['style'] = f"{existing_style} width: 99%;".strip()
    
    return fc.Td(*c, cls=cls_str.strip(), **kwargs) if cls_str.strip() else fc.Td(*c, **kwargs)


def Th(*c,
       shrink = False,
       expand = False,
       cls = (),
       **kwargs):
    """
    Table header cell with MonsterUI-compatible signature.
    
    Args:
        *c: Cell content
        shrink: Minimize cell width
        expand: Maximize cell width
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
        
    Returns:
        Th element
    """
    cls_str = stringify(cls) if cls else ''
    if shrink:
        cls_str += ' no-wrap'
        existing_style = kwargs.get('style', '').strip()
        if existing_style and not existing_style.endswith(';'):
            existing_style += ';'
        kwargs['style'] = f"{existing_style} width: 1%;".strip()
    if expand:
        cls_str += ' no-wrap'
        existing_style = kwargs.get('style', '').strip()
        if existing_style and not existing_style.endswith(';'):
            existing_style += ';'
        kwargs['style'] = f"{existing_style} width: 99%;".strip()
    
    return fc.Th(*c, cls=cls_str.strip(), **kwargs) if cls_str.strip() else fc.Th(*c, **kwargs)


def Thead(*c, cls=(), **kwargs):
    """Table header section."""
    cls_str = stringify(cls) if cls else None
    return fc.Thead(*c, cls=cls_str, **kwargs) if cls_str else fc.Thead(*c, **kwargs)


def Tbody(*c, cls=(), sortable=False, **kwargs):
    """
    Table body section with optional sortable support.
    
    Args:
        *c: Table rows
        cls: Additional CSS classes
        sortable: Add 'sortable' class for SortableJS
        **kwargs: Additional HTML attributes
        
    Returns:
        Tbody element
    """
    cls_str = stringify(cls) if cls else ''
    if sortable:
        cls_str = f"{cls_str} sortable".strip()
    return fc.Tbody(*c, cls=cls_str, **kwargs) if cls_str else fc.Tbody(*c, **kwargs)


def Tfoot(*c, cls=(), **kwargs):
    """Table footer section."""
    cls_str = stringify(cls) if cls else None
    return fc.Tfoot(*c, cls=cls_str, **kwargs) if cls_str else fc.Tfoot(*c, **kwargs)


def TableFromLists(header_data,
                   body_data,
                   footer_data = None,
                   header_cell_render = Th,
                   body_cell_render = Td,
                   footer_cell_render = Td,
                   cls = 'border',
                   sortable = False,
                   **kwargs):
    """
    Create table from lists with MonsterUI-compatible signature.
    
    Args:
        header_data: List of header values
        body_data: List of lists for body rows
        footer_data: Optional list of footer values
        header_cell_render: Function to render header cells
        body_cell_render: Function to render body cells
        footer_cell_render: Function to render footer cells
        cls: Table CSS classes
        sortable: Enable sortable rows
        **kwargs: Additional table attributes
        
    Returns:
        Table component
        
    Example:
        TableFromLists(
            ["Name", "Age"],
            [["Alice", "25"], ["Bob", "30"]],
            footer_data=["Total", "2"]
        )
    """
    return Table(
        Thead(Tr(*map(header_cell_render, header_data))),
        Tbody(*[Tr(*map(body_cell_render, row)) for row in body_data], sortable=sortable),
        Tfoot(Tr(*map(footer_cell_render, footer_data))) if footer_data else None,
        cls=cls,
        **kwargs
    )


def TableFromDicts(header_data,
                   body_data,
                   footer_data = None,
                   header_cell_render = Th,
                   body_cell_render = lambda k, v: Td(v),
                   footer_cell_render = lambda k, v: Td(v),
                   cls = 'border',
                   sortable = False,
                   **kwargs):
    """
    Create table from dicts with MonsterUI-compatible signature.
    
    Args:
        header_data: List of column keys
        body_data: List of dicts for body rows
        footer_data: Optional dict of footer values
        header_cell_render: Function to render header cells
        body_cell_render: Function(key, value) to render body cells
        footer_cell_render: Function(key, value) to render footer cells
        cls: Table CSS classes
        sortable: Enable sortable rows
        **kwargs: Additional table attributes
        
    Returns:
        Table component
        
    Example:
        TableFromDicts(
            ["name", "age"],
            [{"name": "Alice", "age": 25}, {"name": "Bob", "age": 30}],
            footer_data={"name": "Total", "age": "2"}
        )
    """
    return Table(
        Thead(Tr(*[header_cell_render(h) for h in header_data])),
        Tbody(*[Tr(*[body_cell_render(k, row.get(k, '')) for k in header_data]) for row in body_data], sortable=sortable),
        Tfoot(Tr(*[footer_cell_render(k, footer_data.get(k, '')) for k in header_data])) if footer_data else None,
        cls=cls,
        **kwargs
    )


In [ ]:
#| code-fold: true
#| eval: false

def ex_tablelists():
    return Div(
             TableFromLists(
            ["Name", "Age"],
            [["Alice", "25"], ["Bob", "30"]],
            footer_data=["Total", "2"]
        )
    )
preview(ex_tablelists())

In [ ]:
#| code-fold: true
#| eval: false

def ex_tabledicts():
    return Div(
           TableFromDicts(
            ["name", "age"],
            [{"name": "Alice", "age": 25}, {"name": "Bob", "age": 30}],
            footer_data={"name": "Total", "age": "2"}
        )
    )
preview(ex_tabledicts())

In [ ]:
#| export

def TableControls(*controls, cls='', **kwargs):
    """
    Toolbar container for table filters, search, and actions.
    Uses BeerCSS flex layout for horizontal arrangement.
    
    Args:
        *controls: Control elements (inputs, selects, buttons)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        Div with controls arranged horizontally
    
    Examples:
        TableControls(
            Field(Input(placeholder="Search...", hx_get="/table/rows", hx_trigger="keyup changed delay:500ms"), cls="border round"),
            Select("All", "Active", "Inactive", value="All"),
            Button("Add New", cls=ButtonT.primary)
        )
    """
    controls_cls = f"padding middle-align space {cls}".strip()
    return Div(*controls, cls=controls_cls, **kwargs)




In [ ]:
#| code-fold: true
#| eval: false

def ex_tablecontrols():
    return Div(
           TableControls(
            Field(Input(placeholder="Search...", hx_get="/table/rows", hx_trigger="keyup changed delay:500ms"), cls="border round"),
        )
    )
preview(ex_tablecontrols())

## Pagination

In [ ]:
#| export
def Pagination(current_page: int,
               total_pages: int,
               hx_get: str,
               hx_target: str = '#table-container',
               show_first_last: bool = True,
               cls='',
               **kwargs):
    """
    Pagination controls with HTMX integration using BeerCSS native buttons.
    
    Args:
        current_page: Current page number (1-indexed)
        total_pages: Total number of pages
        hx_get: Base URL for pagination requests (e.g., '/table/rows')
        hx_target: HTMX target selector for content swap
        show_first_last: Show first/last page buttons (default: True)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        Nav element with pagination buttons
    
    Examples:
        Pagination(1, 10, '/table/rows')
        Pagination(5, 20, '/api/data', hx_target='#content')
    """
    buttons = []
    
    # Determine separator for query params
    separator = '&' if '?' in hx_get else '?'
    
    # First page button
    if show_first_last:
        first_disabled = current_page == 1
        buttons.append(
            FhButton(
                Icon('first_page'),
                cls='circle transparent' if first_disabled else 'circle',
                disabled=first_disabled,
                hx_get=f"{hx_get}{separator}page=1" if not first_disabled else None,
                hx_target=hx_target,
                hx_push_url='true' if not first_disabled else None
            )
        )
    
    # Previous page button
    prev_disabled = current_page == 1
    buttons.append(
        FhButton(
            Icon('chevron_left'),
            cls='circle transparent' if prev_disabled else 'circle',
            disabled=prev_disabled,
            hx_get=f"{hx_get}{separator}page={current_page - 1}" if not prev_disabled else None,
            hx_target=hx_target,
            hx_push_url='true' if not prev_disabled else None
        )
    )
    
    # Page indicator
    buttons.append(
        Span(f"Page {current_page} of {total_pages}", cls='small-text')
    )
    
    # Next page button
    next_disabled = current_page >= total_pages
    buttons.append(
        FhButton(
            Icon('chevron_right'),
            cls='circle transparent' if next_disabled else 'circle',
            disabled=next_disabled,
            hx_get=f"{hx_get}{separator}page={current_page + 1}" if not next_disabled else None,
            hx_target=hx_target,
            hx_push_url='true' if not next_disabled else None
        )
    )
    
    # Last page button
    if show_first_last:
        last_disabled = current_page >= total_pages
        buttons.append(
            FhButton(
                Icon('last_page'),
                cls='circle transparent' if last_disabled else 'circle',
                disabled=last_disabled,
                hx_get=f"{hx_get}{separator}page={total_pages}" if not last_disabled else None,
                hx_target=hx_target,
                hx_push_url='true' if not last_disabled else None
            )
        )
    
    nav_cls = f"center-align middle-align {cls}".strip()
    return Nav(*buttons, cls=nav_cls, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_tablePagination():
    return Div(
                 Pagination(1, 10, '/table/rows')
    )
preview(ex_tablePagination())

## Card

In [ ]:
#| export

def Card(*c,
         header = None,
         footer = None,
         body_cls = 'padding',
         header_cls = (),
         footer_cls = (),
         cls = (),
         **kwargs):
    """
    A Card component using BeerCSS native elements.
    
    Args:
        *c: Child content (body content if header/footer not specified)
        header: Optional header content
        footer: Optional footer content
        body_cls: Classes for body div (default 'padding')
        header_cls: Classes for header element
        footer_cls: Classes for footer/nav element
        cls: Classes for article container
        **kwargs: Additional attributes for article
        
    Returns:
        Article element with header, body, and/or footer sections
        
    Example:
        Card("Body content", header="Title", cls="border round")
        Card(
            "Main content here",
            header=H5("Card Title"),
            footer=Button("Action"),
            cls="border round surface-container"
        )
    """
    # Normalize classes
    cls = normalize_tokens(cls)
    header_cls = normalize_tokens(header_cls)
    footer_cls = normalize_tokens(footer_cls)
    body_cls = normalize_tokens(body_cls)
    
    # Build sections
    sections = []
    
    if header is not None:
        sections.append(Header(header, cls=header_cls) if header_cls else Header(header))
    
    if c:
        sections.append(Div(*c, cls=body_cls) if body_cls else Div(*c))
    
    if footer is not None:
        sections.append(Nav(footer, cls=footer_cls) if footer_cls else Nav(footer))
    
    return Article(*sections, cls=cls, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_card():
    return Div(
      Card("Body content", header="Title", cls="border round"),
        Card(
            "Main content here",
            header=H5("Card Title"),
            footer=Button("Action"),
            cls="border round surface-container"
        )
    )
preview(ex_card())

## Toolbar

In [ ]:
#| export

def Toolbar(*items, cls='', elevate='large', fill=True, **kwargs):
    """
    BeerCSS Toolbar component for action bars.
    
    Args:
        *items: Child elements (typically A elements with icons)
        cls: Additional CSS classes
        elevate: Elevation level ('small', 'medium', 'large', or None)
        fill: Whether toolbar should have fill effect (default: True)
        **kwargs: Additional HTML attributes
    
    Example:
        Toolbar(
            A(Icon('videocam_off')),
            A(Icon('mic')),
            A(Icon('front_hand'), cls='active'),
            A(Icon('more_vert'))
        )
    
    Returns:
        Nav element with toolbar classes
    """
    classes = ['toolbar']
    
    if elevate:
        classes.append(f'{elevate}-elevate')
    
    if fill:
        classes.append('fill')
    
    if cls:
        classes.append(cls)
    
    final_cls = ' '.join(classes)
    
    return Nav(*items, cls=final_cls, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_toolbar():
    return Div(
       Toolbar(
            A(Icon('videocam_off')),
            A(Icon('mic')),
            A(Icon('front_hand'), cls='active'),
            A(Icon('more_vert'))
        )
    )
preview(ex_toolbar())

## Toast / Snackbar

In [ ]:
#| export

def Toast(*c, cls='', position='top', variant='', action=None, dur=5.0, active=False, **kwargs):
    """
    BeerCSS Snackbar/Toast component for notifications.
    
    Args:
        *c: Child content (text or elements)
        cls: Additional CSS classes
        position: Position of toast ('top', 'bottom', 'left', 'right')
        variant: Toast variant ('error', 'success', 'warning', '' for default)
        action: Optional action link text or element
        dur: Duration in seconds (for auto-dismiss, not implemented in CSS)
        active: Whether toast is active/visible
        **kwargs: Additional HTML attributes
    
    Example:
        Toast("Some text here", position='top', active=True)
        Toast("Error occurred", variant='error', position='top', active=True)
        Toast("File saved", action=A("Undo", cls='inverse-link'), position='bottom')
    
    Returns:
        Div element with snackbar classes
    """
    classes = ['snackbar']
    
    if variant:
        classes.append(variant)
    
    # BeerCSS position classes are reversed - 'top' class renders at bottom, etc.
    if position:
        position_map = {'top': 'bottom', 'bottom': 'top', 'left': 'right', 'right': 'left'}
        classes.append(position_map.get(position, position))
    
    if active:
        classes.append('active')
    
    if cls:
        classes.append(cls)
    
    final_cls = ' '.join(classes)
    
    # Build content
    content = []
    
    # If action provided, wrap content in max div
    if action:
        if c:
            content.append(Div(*c, cls='max'))
        if isinstance(action, str):
            content.append(A(action, cls='inverse-link'))
        else:
            content.append(action)
    else:
        content.extend(c)
    
    return Div(*content, cls=final_cls, **kwargs)


def Snackbar(*c, **kwargs):
    """Alias for Toast component"""
    return Toast(*c, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_toast():
    return Div(
       Toast("Some text here",  variant='error', position='top', action=A("Undo", cls='inverse-link'), active=True),
      #  Toast("Error occurred", variant='error', position='top', active=True),
       # Toast("File saved", action=A("Undo", cls='inverse-link'), position='bottom')
    )
preview(ex_toast())

### Others

In [ ]:
#| export

def Divider(cls='', **kwargs):
    """Horizontal divider line"""
    return Hr(cls=cls, **kwargs)

def DividerSplit(text='', text_cls='', cls='', **kwargs):
    """Divider with centered text"""
    return Div(
        Span(text, cls=f"small-text {text_cls}".strip()),
        cls=f"center-align {cls}".strip(),
        style="position: relative; margin: 1rem 0;",
        **kwargs
    )

def Avatar(name='', src='', size=10, cls='', **kwargs):
    """Simple avatar component - circular image or initial"""
    if src:
        return Img(src=src, alt=name, cls=f"circle {cls}".strip(), 
                   style=f"width: {size}rem; height: {size}rem; object-fit: cover;", **kwargs)
    else:
        initial = name[0].upper() if name else '?'
        return Div(
            initial,
            cls=f"circle center-align middle-align {cls}".strip(),
            style=f"width: {size}rem; height: {size}rem; background: var(--primary); color: var(--on-primary); font-size: {size/2}rem; line-height: {size}rem;",
            **kwargs
        )

def Subtitle(text, cls='', **kwargs):
    """Muted subtitle text"""
    return P(text, cls=f"small-text {cls}".strip(), **kwargs)

## Nav Container

In [ ]:
#| export

def NavContainer(*li, 
                 title=None,
                 brand=None,
                 position='left',
                 close_button=True,
                 cls='active',
                 id=None,
                 **kwargs):
    """
    Dialog-based navigation container using BeerCSS structure.
    Responsive by default - works as drawer on mobile, can be sidebar on desktop.
    
    Args:
        *li: Navigation items (Li tags with icon and text)
        title: Title text for header (H6)
        brand: Brand/logo image (path string or Img element)
        position: 'left', 'right', 'top', 'bottom' (default: 'left')
        close_button: Include close button in header (default: True)
        cls: Additional CSS classes (default: 'active')
        id: Dialog ID (required for data-ui triggers)
        **kwargs: Additional HTML attributes
    
    Returns:
        Dialog element with BeerCSS position classes
    
    Examples:
        # Basic dialog navigation (items auto-styled with 'wave round')
        NavContainer(
            Li(I('inbox'), Span('Inbox', cls='max'), B('24')),
            Li(I('send'), Span('Outbox')),
            Li(I('favorite'), Span('Favorites')),
            title='Title',
            brand='/favicon.png',
            id='main-nav'
        )
        
        # With custom brand image
        NavContainer(
            Li(I('home'), Span('Home')),
            Li(I('settings'), Span('Settings')),
            title='My App',
            brand=Img(src='/logo.png', cls='circle large'),
            id='sidebar'
        )
        
        # Trigger with button
        Button(I('menu'), data_ui='#main-nav')
    """
    children = []
    
    # Header with nav structure
    header_nav_content = []
    
    # Add brand image if provided
    if brand:
        if isinstance(brand, str):
            # Brand is a path string
            header_nav_content.append(Img(src=brand, cls='circle large'))
        else:
            # Brand is an element
            header_nav_content.append(brand)
    
    # Add title if provided
    if title:
        header_nav_content.append(H6(title, cls='max'))
    
    # Add close button if requested
    if close_button and id:
        header_nav_content.append(
            FhButton(I('close'), cls='transparent circle large', data_ui=f'#{id}')
        )
    
    # Wrap header content in nav, then header
    if header_nav_content:
        children.append(Header(Nav(*header_nav_content)))
    
    # Add spacer div
    children.append(Div(cls='space'))
    
    # Auto-apply 'wave round' to Li items that don't have cls already
    processed_items = []
    for item in li:
        if hasattr(item, 'tag') and item.tag == 'li':
            # Check if item already has cls attribute
            item_attrs = getattr(item, 'attrs', {})
            item_cls = item_attrs.get('cls', '')
            if not item_cls:
                # Clone item with default classes
                new_attrs = dict(item_attrs)
                new_attrs['cls'] = 'wave round'
                processed_items.append(Li(*item.children, **new_attrs))
            else:
                processed_items.append(item)
        else:
            processed_items.append(item)
    
    # Wrap navigation items in ul.list
    if processed_items:
        children.append(Ul(*processed_items, cls='list'))
    
    # Build dialog classes
    dialog_cls = [position]
    if cls:
        dialog_cls.append(cls)
    
    cls_str = ' '.join(dialog_cls)
    
    if id:
        return Dialog(*children, id=id, cls=cls_str, **kwargs)
    return Dialog(*children, cls=cls_str, **kwargs)


def NavHeaderLi(*c, cls='horizontal-padding', **kwargs):
    """
    Navigation header as list item (Monster UI signature).
    
    Args:
        *c: Header content (title, brand, icons, etc.)
        cls: CSS classes (default: 'horizontal-padding')
        **kwargs: Additional HTML attributes
    
    Returns:
        Header element for navigation
    
    Examples:
        NavHeaderLi(H5('Sources'))
        NavHeaderLi(
            DivLAligned(
                Icon('folder_open', cls='primary-text'),
                H5('My Files')
            )
        )
    """
    return Header(*c, cls=cls, **kwargs)


def NavDividerLi(cls='', **kwargs):
    """
    Navigation divider as list item (Monster UI signature).
    
    Args:
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        Hr element for visual separation
    
    Example:
        NavDividerLi()
        NavDividerLi(cls='large-margin')
    """
    return Hr(cls=cls, **kwargs)


def NavCloseLi(dialog_id: str, icon='close', cls='circle transparent', **kwargs):
    """
    Navigation close button as list item (Monster UI signature).
    
    Args:
        dialog_id: ID of dialog to close
        icon: Material icon name (default: 'close')
        cls: Button CSS classes (default: 'circle transparent')
        **kwargs: Additional HTML attributes
    
    Returns:
        Button element with close functionality
    
    Examples:
        NavCloseLi('main-nav')
        NavCloseLi('sidebar', icon='arrow_back')
    """
    return FhButton(I(icon), cls=cls, onclick=f"closeDialog('{dialog_id}')", **kwargs)


def NavSubtitle(*c, cls='small-text gray-text padding', **kwargs):
    """
    Navigation section subtitle (Monster UI signature).
    
    Args:
        *c: Subtitle text or elements
        cls: CSS classes (default: 'small-text gray-text padding')
        **kwargs: Additional HTML attributes
    
    Returns:
        Label element for section headers
    
    Examples:
        NavSubtitle('DOCUMENTS')
        NavSubtitle('Settings', cls='small-text primary-text padding')
    """
    return Label(*c, cls=cls, **kwargs)


def BottomNav(*c, cls='bottom', size='s', **kwargs):
    """
    Mobile bottom navigation bar (BeerCSS specific).
    
    Args:
        *c: Navigation links (A tags with Icon + Span)
        cls: CSS classes (default: 'bottom')
        size: Size variant ('s', 'm', 'l') (default: 's')
        **kwargs: Additional HTML attributes
    
    Returns:
        Nav element with bottom navigation styling
    
    Examples:
        BottomNav(
            A(Icon('home'), Span('Home'), href='/'),
            A(Icon('search'), Span('Search'), href='/search'),
            A(Icon('notifications'), Span('Alerts'), href='/alerts'),
            A(Icon('person'), Span('Profile'), href='/profile')
        )
    """
    nav_cls = f"{cls} {size}".strip()
    return Nav(*c, cls=nav_cls, **kwargs)

In [ ]:
#| code-fold: true
#| eval: false

def ex_navcontainer():
    return NavContainer(
        Li(I('inbox'), Span('Inbox', cls='max'), B('24')),
        Li(I('send'), Span('Outbox')),
        Li(I('favorite'), Span('Favorites')),
        Li(I('delete'), Span('Trash')),
        Li(I('fiber_manual_record'), Span('Label')),
        Li(I('change_history'), Span('Label')),
        Li(I('stop'), Span('Label')),
        title='Title',
        brand='/favicon.png',
        id='main-nav'

    )

preview(ex_navcontainer())

## Sidebar

In [ ]:
#| export

def NavSideBarHeader(*c, cls='', **kwargs):
    """
    Navigation Sidebar header section for menu buttons and branding.
    
    Args:
        *c: Child elements (buttons, icons, etc.)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        Header element for use within NavSideBarContainer
    
    Example:
        NavSideBarHeader(
            Button(I('menu'), cls='circle transparent'),
            Button(I('widgets'), Span('Explore'), cls='square round')
        )
    """
    return Header(*c, cls=cls, **kwargs)


def NavSideBarLinks(*children, as_list=False, cls='', **kwargs):
    """
    Container for navigation links supporting multiple styles.
    
    Args:
        *children: A() links, Li() items, or other content
        as_list: If True, wraps in <ul class="list"> for BeerCSS list styling
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        Either <ul class="list"> or Group of children
    
    Examples:
        # Direct links (no list wrapper)
        NavSideBarLinks(
            A(I('home'), Span('Home')),
            A(I('settings'), Span('Settings'))
        )
        
        # List-style with border
        NavSideBarLinks(
            Li(A('Home')),
            Li(A('Settings')),
            as_list=True,
            cls='border'
        )
        
        # Nested/expandable sections
        NavSideBarLinks(
            Li(A('Home')),
            Li(Details(
                Summary('Settings'),
                Ul(Li('Profile'), Li('Security'), cls='list border')
            )),
            as_list=True
        )
    """
    if as_list:
        list_cls = f"list {cls}".strip()
        return Ul(*children, cls=list_cls, **kwargs)
    else:
        # Return children directly without wrapper
        return Group(*children) if len(children) > 1 else (children[0] if children else Group())


def NavSideBarContainer(*children, position='left', size='m', cls='', active=False, **kwargs):
    """
    BeerCSS Navigation Sidebar/Drawer component with surface-container background.
    
    Args:
        *children: NavSideBarHeader, NavSideBarLinks, or direct A() elements
        position: 'left', 'right', 'top', 'bottom' (default: 'left')
        size: 's', 'm', 'l' (default: 'm')
        cls: Additional CSS classes
        active: If True, nav is visible by default (default: False - starts collapsed)
        **kwargs: Additional HTML attributes (include 'id' for toggle functionality)
    
    Returns:
        <nav> element with BeerCSS navigation classes and surface-container background
    
    Note:
        By default, navigation starts collapsed. Add a toggle button with 
        data_ui="#nav-id" to expand/collapse it.
    
    Examples:
        # Nav with toggle (default behavior)
        NavSideBarContainer(
            NavSideBarHeader(
                Button(I('menu'), data_ui="#my-nav")
            ),
            A(I('home'), Span('Home')),
            A(I('settings'), Span('Settings')),
            position='left',
            id='my-nav'
        )
        
        # Nav always visible
        NavSideBarContainer(
            A(I('home'), Span('Home')),
            A(I('settings'), Span('Settings')),
            position='left',
            active=True
        )
    """
    base_cls = f"{size} {position} surface-container"
    if active:
        base_cls += " active"
    nav_cls = f"{base_cls} {cls}".strip()
    return Nav(*children, cls=nav_cls, **kwargs)


In [ ]:
#| code-fold: true
#| eval: false

def ex_navsidebarcontainer():
         
        nav = NavSideBarContainer(
            NavSideBarHeader(
                NavToggleButton('#main-nav')
            ),
            A(I('home'), Span('Home')),
            A(I('dashboard'), Span('Dashboard')),
            A(I('settings'), Span('Settings')),
            A(I('help'), Span('Help')),
            A(I('info'), Span('About')),
            position='left',
            size='l',
            id='main-nav'
        )
        
        content = Main(
            H2("Navigation Toggle"),
            H3("Content Area"),
            P("Click the menu button to toggle the navigation."),
            cls="responsive"
        )
    
        return Div(nav, content)


preview(ex_navsidebarcontainer())

In [ ]:
#| export

def Layout(*content,
           sidebar=None,
           sidebar_links=None,
           nav_bar=None,
           container_size=ContainerT.expand,
           main_bg='surface',
           sidebar_id='app-sidebar',
           cls='',
           **kwargs):
    """
    Standard app layout wrapper with auto-toggle sidebar and sensible defaults.
    
    Args:
        *content: Main content elements (will be wrapped in Container)
        sidebar: Optional sidebar content (links, etc.) - auto-wrapped with toggle header
        sidebar_links: Alternative to sidebar - just provide the links, header+toggle added automatically
        nav_bar: Optional NavBar for top navigation (sticky + full-width by default)
        container_size: ContainerT size for main content (default: expand for full-width)
        main_bg: Background class for main content area (default: 'surface')
        sidebar_id: ID for sidebar toggle functionality (default: 'app-sidebar')
        cls: Additional CSS classes for outer wrapper
        **kwargs: Additional HTML attributes for outer wrapper
    
    Returns:
        Layout structure with sidebar and main content properly positioned
    
    Defaults:
        - Sidebar: Auto-includes NavRailHeader with toggle button
        - Container: Expands to full width (ContainerT.expand)
        - NavBar: Sticky and full-width by default
        - Main content: Has neutral background color
    
    Example:
        Layout(
            H1("Dashboard"),
            P("Main content here"),
            sidebar_links=[
                A(I('home'), Span('Home')),
                A(I('settings'), Span('Settings'))
            ],
            nav_bar=NavBar(
                A("Home", href='/'),
                A("About", href='/about'),
                brand=H3('My App')
            )
        )
    """
    # Build main content area
    main_content = []
    
    # Add nav bar if provided (sticky + full-width by default)
    if nav_bar:
        # Ensure navbar is sticky
        if hasattr(nav_bar, 'attrs') and 'cls' in nav_bar.attrs:
            if 'sticky' not in nav_bar.attrs['cls']:
                nav_bar.attrs['cls'] += ' sticky top'
        main_content.append(nav_bar)
    
    # Wrap content in Container with background
    if content:
        container_cls = (container_size, 'padding', main_bg)
        main_content.append(Container(*content, cls=container_cls))
    
    # If no sidebar, return simple structure
    if not sidebar and not sidebar_links:
        if main_content:
            return Div(*main_content, cls=cls, **kwargs)
        else:
            return Div(cls=cls, **kwargs)
    
    # Build sidebar with auto-toggle header
    sidebar_children = []
    
    # Auto-add header with toggle button (uses Icon helper)
    sidebar_children.append(
        NavSideBarHeader(NavToggleButton(f"#{sidebar_id}"))
        )
    
    
    # Add sidebar content
    if sidebar_links:
        sidebar_children.extend(sidebar_links)
    elif sidebar:
        if is_listy(sidebar):
            sidebar_children.extend(sidebar)
        else:
            sidebar_children.append(sidebar)
    
    # Create sidebar with auto-toggle
    nav_rail = NavSideBarContainer(
        *sidebar_children,
        position='left',
        size='l',
        id=sidebar_id
    )
    
    # Separate navbar from main content so it spans full width
    navbar_elem = None
    content_items = []
    for item in main_content:
        if hasattr(item, 'tag') and item.tag == 'nav':
            navbar_elem = item
        else:
            content_items.append(item)
    
    # Build layout with navbar spanning full width over sidebar
    layout_children = []
    if navbar_elem:
        layout_children.append(navbar_elem)  # Navbar first, full width
    layout_children.append(nav_rail)  # Sidebar
    if content_items:
        # Use Container to respect container_size parameter and maintain consistent styling
        container_cls = (container_size, 'round', 'elevate', 'margin')
        layout_children.append(Container(*content_items, cls=container_cls))
    
    # Use surface-container background to match navbar and sidebar
    final_cls = f"surface-container {cls}".strip() if cls else "surface-container"
    return Div(*layout_children, cls=final_cls, **kwargs)

## Typography

In [ ]:
#| export

class TextT(VEnum):
    """Text styles using BeerCSS typography classes"""
    
    # Text formatting
    italic = 'italic'
    bold = 'bold'
    underline = 'underline'
    overline = 'overline'
    upper = 'upper'
    lower = 'lower'
    capitalize = 'capitalize'
    
    # Text sizes
    small_text = 'small-text'
    medium_text = 'medium-text'
    large_text = 'large-text'
    
    # Sizes (for elements)
    small = 'small'
    medium = 'medium'
    large = 'large'
    
    # Line spacing
    no_line = 'no-line'
    tiny_line = 'tiny-line'
    small_line = 'small-line'
    medium_line = 'medium-line'
    large_line = 'large-line'
    extra_line = 'extra-line'
    
    # Links
    link = 'link'
    inverse_link = 'inverse-link'
    
    # Alignment
    left_align = 'left-align'
    right_align = 'right-align'
    center_align = 'center-align'
    
    # Text colors
    primary_text = 'primary-text'
    secondary_text = 'secondary-text'
    tertiary_text = 'tertiary-text'


class TextPresets(VEnum):
    """Common typography presets combining multiple TextT values"""
    
    muted_sm = 'small-text secondary-text'
    muted_lg = 'large-text secondary-text'
    bold_sm = 'bold small-text'
    bold_lg = 'bold large-text'
    medium_sm = 'medium small-text'
    medium_muted = 'medium secondary-text'
    primary_link = 'link primary-text'
    muted_link = 'link secondary-text'

In [ ]:
#| export

def CodeSpan(*c, cls=(), **kwargs):
    """
    Inline code snippet.
    
    Args:
        *c: Contents (text)
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <code> element for inline code
    
    Example:
        CodeSpan("print('hello')")
    """
    cls_str = stringify(cls) if cls else None
    return Code(*c, cls=cls_str, **kwargs) if cls_str else Code(*c, **kwargs)


def CodeBlock(*c, cls=(), code_cls=(), **kwargs):
    """
    Block code with pre wrapper.
    
    Args:
        *c: Code content
        cls: CSS classes for pre wrapper
        code_cls: CSS classes for code element
        **kwargs: Additional HTML attributes for code element
    
    Returns:
        <pre><code> block
    
    Example:
        CodeBlock("def hello():\n    print('world')")
    """
    code_cls_str = stringify(code_cls) if code_cls else None
    pre_cls_str = stringify(cls) if cls else None
    
    code_elem = Code(*c, cls=code_cls_str, **kwargs) if code_cls_str else Code(*c, **kwargs)
    return Pre(code_elem, cls=pre_cls_str) if pre_cls_str else Pre(code_elem)


def Blockquote(*c, cls=(), **kwargs):
    """
    Blockquote element.
    
    Args:
        *c: Quote content
        cls: Additional CSS classes
        **kwargs: Additional HTML attributes
    
    Returns:
        <blockquote> element
    
    Example:
        Blockquote("To be or not to be")
    """
    cls_str = stringify(cls) if cls else None
    return fc.Blockquote(*c, cls=cls_str, **kwargs) if cls_str else fc.Blockquote(*c, **kwargs)


def Q(*c, cls='italic large-text', **kwargs):
    """Styled quotation with italic and large text"""
    return fc.Q(*c, cls=cls, **kwargs)


def Em(*c, cls=(), **kwargs):
    """Emphasized text"""
    cls_str = stringify(cls) if cls else None
    return fc.Em(*c, cls=cls_str, **kwargs) if cls_str else fc.Em(*c, **kwargs)


def Strong(*c, cls='bold', **kwargs):
    """Strong (bold) text"""
    return fc.Strong(*c, cls=cls, **kwargs)


def Small(*c, cls='small-text', **kwargs):
    """Small text"""
    return fc.Small(*c, cls=cls, **kwargs)


def Mark(*c, cls=(), **kwargs):
    """Highlighted/marked text"""
    cls_str = stringify(cls) if cls else None
    return fc.Mark(*c, cls=cls_str, **kwargs) if cls_str else fc.Mark(*c, **kwargs)


def Del(*c, cls='secondary-text', **kwargs):
    """Deleted/strikethrough text with muted color"""
    return fc.Del(*c, cls=cls, **kwargs)


def Ins(*c, cls='underline', **kwargs):
    """Inserted/underlined text"""
    return fc.Ins(*c, cls=cls, **kwargs)


def Sub(*c, cls='small-text', **kwargs):
    """Subscript text"""
    return fc.Sub(*c, cls=cls, **kwargs)


def Sup(*c, cls='small-text', **kwargs):
    """Superscript text"""
    return fc.Sup(*c, cls=cls, **kwargs)

## FAQ

In [ ]:
#| export
def FAQItem(question: str, answer: str, question_cls: str = '', answer_cls: str = ''):
    """Render one FAQ item using BeerCSS-ish `details/summary` markup."""
    return Details(
        Summary(
            Article(
                Nav(
                    Div(question, cls=f"max bold {question_cls}".strip()),
                    I("expand_more"),
                ),
                cls="round primary no-elevate",
            )
        ),
        Article(
            P(answer, cls=f"secondary-text {answer_cls}".strip()),
            cls="round border padding",
        ),
    )


#| export
def FAQList(title: str, faqs: list, cls: str = ''):
    """Render a list of FAQs.

    `faqs` must be a list of dicts with keys: `question`, `answer`.
    """
    if not isinstance(faqs, list):
        raise TypeError("FAQList: `faqs` must be a list of dicts")

    items = []
    for i, faq in enumerate(faqs):
        if not isinstance(faq, dict):
            raise TypeError(f"FAQList: faqs[{i}] must be a dict")
        if "question" not in faq or "answer" not in faq:
            raise KeyError(f"FAQList: faqs[{i}] must have keys 'question' and 'answer'")
        items.append(FAQItem(faq["question"], faq["answer"]))

    items_container = Div(
        *items,
        style="display:flex; flex-direction:column; gap:1rem;",
    )

    return Section(
        H3(title, cls="medium"),
        items_container,
        cls=f"responsive {cls}".strip(),
    )

In [ ]:
#| eval: false
def ex_faq_list():
    faqs = [
        {"question": "What is fh-matui?", "answer": "A small component layer for FastHTML."},
        {"question": "Does FAQList accept tuples?", "answer": "No — it is strict and only accepts dicts."},
    ]
    return FAQList("FAQ", faqs)

In [ ]:
preview(ex_faq_list())

## Cookies Banner

In [ ]:
#| export

def CookiesBanner(
    message='We use cookies to enhance your experience. By continuing to visit this site you agree to our use of cookies.',
    accept_text='Accept',
    decline_text='Decline',
    settings_text=None,
    policy_link='/cookies',
    policy_text='Learn more',
    position='bottom',
    on_accept='console.log("Accepted")',
    on_decline='console.log("Declined")',
    on_settings='console.log("Settings")',
    cls='',
    **kwargs
):
    """
    Reusable cookie consent banner component.
    
    Args:
        message: Cookie policy message (default: standard message)
        accept_text: Accept button text (default: 'Accept')
        decline_text: Decline button text (default: 'Decline')
        settings_text: Optional settings button text (default: None)
        policy_link: Cookie policy page link (default: '/cookies')
        policy_text: Policy link text (default: 'Learn more')
        position: Banner position - 'top', 'bottom' (default: 'bottom')
        on_accept: JavaScript for accept (default: console.log)
        on_decline: JavaScript for decline (default: console.log)
        on_settings: JavaScript for settings (default: console.log)
        cls: Additional CSS classes
        **kwargs: Additional attributes
    
    Returns:
        Div component styled as cookie banner
    
    Examples:
        CookiesBanner()
        CookiesBanner(message='We use cookies', position='top')
        CookiesBanner(settings_text='Customize', on_accept='acceptCookies()')
    """
    # Build message with policy link
    message_content = Div(
        Span(message, cls='small-text'),
        ' ',
        A(policy_text, href=policy_link, cls=AT.primary),
        cls='max'
    )
    
    # Build action buttons
    buttons = []
    
    if decline_text:
        buttons.append(
            Button(
                decline_text,
                type='button',
                cls=ButtonT.secondary,
                onclick=f"{on_decline}; this.closest('.cookie-banner').remove();"
            )
        )
    
    if settings_text:
        buttons.append(
            Button(
                settings_text,
                type='button',
                cls=ButtonT.secondary,
                onclick=on_settings
            )
        )
    
    buttons.append(
        Button(
            accept_text,
            type='button',
            cls=ButtonT.primary,
            onclick=f"{on_accept}; this.closest('.cookie-banner').remove();"
        )
    )
    
    # Build banner content
    banner_content = Div(
        Icon('cookie', cls='medium'),
        message_content,
        Div(*buttons, cls='row'),
        cls='row middle-align'
    )
    
    # Position styling
    position_style = {
        'top': 'position: fixed; top: 0; left: 0; right: 0; z-index: 9999;',
        'bottom': 'position: fixed; bottom: 0; left: 0; right: 0; z-index: 9999;'
    }
    
    style = position_style.get(position, position_style['bottom'])
    
    # Build banner
    banner_cls = f'cookie-banner surface-container padding shadow {cls}'.strip()
    
    return Div(
        banner_content,
        cls=banner_cls,
        style=style,
        **kwargs
    )

In [ ]:
#| code-fold: true
#| eval: false

def get_CookiesBanner():
    return CookiesBanner()

preview(get_CookiesBanner())

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()